# Vapor Policy Impact — Databricks Walkthrough

| | |
|---|---|
| **Purpose** | Estimate the impact of a state-level vapor policy on tobacco retail volume, before it is implemented in a new state |
| **Methodology reference** | [`Vapor-Policy-Impact-Forecasting-Methodology.md`](../Vapor-Policy-Impact-Forecasting-Methodology.md) |
| **Package reference** | [`README.md`](../README.md) |
| **Estimated runtime** | ~5 minutes with default settings; ~15-20 minutes with leave-one-state-out validation enabled (Part 8) |
| **Data** | Synthetic by default, no setup required — see **Part 2 · Configuration** to point at your own data |

> **Synthetic vs. real data.** By default this notebook runs against a synthetic,
> simulated panel with a known, injected policy effect (no real retail-scan data was
> available when this framework was built, so every component was validated against
> ground truth first). Flip `USE_SYNTHETIC_DATA = False` in **Part 2 · Configuration**
> and fill in your own data source to run against real data — every part after that is
> unchanged either way; nothing downstream cares where the panel came from.

> **Read this before trusting a number.** Every simplification relative to the full
> methodology (the event-study estimator is a from-scratch simplified
> Callaway–Sant'Anna, the "BSTS" baseline is a frequentist state-space fit via
> `statsmodels`, etc.) is documented in the relevant module's docstring in
> `vapor_policy_impact/`. Leave-one-state-out validation during development showed
> ~100% sign accuracy but weak rank-correlation on effect *magnitude* with only 9
> historical treated states — expect similar caveats on your own data, and lean on
> **Part 8 · Model Validation** rather than trusting a single point estimate.


## Contents

- [Part 1 · Environment Setup](#part-1-environment-setup)
- [Part 2 · Configuration](#part-2-configuration)
- [Part 3 · Load Data](#part-3-load-data)
- [Part 4 · Exploratory Checks](#part-4-exploratory-checks)
- [Part 5 · Causal Modeling & 13-Week Forecast](#part-5-causal-modeling-13-week-forecast)
- [Part 6 · Cross-Category Substitution](#part-6-cross-category-substitution)
- [Part 7 · Altria vs. Competitor Decomposition](#part-7-altria-vs-competitor-decomposition)
- [Part 8 · Model Validation](#part-8-model-validation)
- [Part 9 · Executive Summary & Productionizing on Databricks](#part-9-executive-summary-productionizing-on-databricks)

> **Note on the links above:** GitHub and Jupyter resolve these anchor links automatically from the headers below; Databricks' notebook viewer does not consistently support in-notebook anchor navigation, so treat this as a reading guide there rather than clickable navigation.


---

## Part 1 · Environment Setup

Installs dependencies, locates the `vapor_policy_impact` package, and sets up two small display helpers used throughout this notebook. Nothing here is specific to your data — run it once per session and move on to **Part 2**.


### 1.1 Install Dependencies


In [ ]:
%pip install -q "numpy>=1.26" "pandas>=2.1" "scipy>=1.11" "statsmodels>=0.14" "scikit-learn>=1.3" "matplotlib>=3.8"

In [ ]:
try:
    dbutils.library.restartPython()  # noqa: F821 -- only defined on Databricks
except NameError:
    pass

### 1.2 Locate the Package and Set Up Display Helpers

Works whether this notebook was opened via **Databricks Repos** (clone this branch as a Repo — the package sits right next to this notebook) or a local Jupyter checkout. If auto-discovery fails for your setup, set `REPO_ROOT_OVERRIDE` below explicitly.


In [ ]:
import os
import sys


def _find_repo_root(start: str, marker: str = "vapor_policy_impact", max_up: int = 6) -> str:
    d = os.path.abspath(start)
    for _ in range(max_up):
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return start


REPO_ROOT_OVERRIDE = None  # e.g. "/Workspace/Repos/you@company.com/Data-Science--Cheat-Sheet"
REPO_ROOT = REPO_ROOT_OVERRIDE or _find_repo_root(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

print(f"Using REPO_ROOT = {REPO_ROOT}")
assert os.path.isdir(os.path.join(REPO_ROOT, "vapor_policy_impact")), (
    "Could not find the vapor_policy_impact package automatically. Set REPO_ROOT_OVERRIDE "
    "above to the checked-out repo path (e.g. your Databricks Repos folder for this branch)."
)

In [ ]:
# Databricks defines `display()` globally with a rich native table/plot UI; fall back to
# IPython's for local Jupyter so every display(...) call below works in both places.
try:
    display  # noqa: F821
except NameError:
    from IPython.display import display


def show_figure(fig):
    """Render a matplotlib Figure as an actual inline image, in Databricks or Jupyter.

    Passing a bare Figure to display() only shows its plain-text repr
    ("<Figure size ... with 1 Axes>") unless the notebook's kernel has separately
    registered a PNG formatter for it (e.g. via the `%matplotlib inline` magic) --
    not guaranteed across environments, and vapor_policy_impact.reporting.visuals
    deliberately forces the non-interactive Agg backend. Rendering to PNG bytes
    ourselves and wrapping in IPython.display.Image sidesteps that entirely and
    works the same way everywhere.
    """
    import io

    from IPython.display import Image

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    buf.seek(0)
    display(Image(buf.read()))


def style_executive_table(df):
    """Presentation-only formatting for an executive table: percentage-format the
    '% Impact' column and color negative/positive values red/green. The underlying
    build_executive_table() DataFrame stays plain numeric for programmatic use --
    this is purely for how it's displayed in this notebook.
    """
    def _color_sign(v):
        try:
            return "color: #b00020" if v < 0 else ("color: #1a7f37" if v > 0 else "")
        except TypeError:
            return ""

    numeric_fmt = {
        "No Policy": "{:,.2f}",
        "Policy": "{:,.2f}",
        "Incremental Impact": "{:+,.2f}",
        "% Impact": "{:+.1%}",
    }
    # Styler.map/.apply require a unique index -- build_executive_table() output is
    # fine on its own, but concatenating two of them (e.g. the Altria/Competitor
    # decomposition table in Part 7) produces duplicate index labels. Reset it since
    # we hide the index from display anyway.
    df = df.reset_index(drop=True)
    styled = df.style.format(numeric_fmt).map(_color_sign, subset=["Incremental Impact", "% Impact"])
    return styled.hide(axis="index")


def display_executive_table(df):
    """display() an executive table with the styling above where the environment
    supports rich HTML rendering, falling back to a plain table otherwise.
    """
    try:
        display(style_executive_table(df))
    except Exception:
        display(df)

### 1.3 Framework Imports


In [ ]:
import numpy as np
import pandas as pd

from vapor_policy_impact.config import CATEGORIES
from vapor_policy_impact.data.simulate import simulate_panel
from vapor_policy_impact.data.loaders import (
    date_to_week_index,
    load_policy_calendar,
    load_real_panel,
    load_state_covariates,
)
from vapor_policy_impact.pipeline import prepare_covariates, run_category_pipeline
from vapor_policy_impact.decomposition.manufacturer import decompose_manufacturer
from vapor_policy_impact.substitution.cross_category import category_residual_correlation, reconcile_categories
from vapor_policy_impact.features.engineering import build_aggregated_feature_series
from vapor_policy_impact.reporting.business_output import build_executive_table
from vapor_policy_impact.reporting.visuals import (
    plot_event_study,
    plot_scenario_fan_chart,
    plot_share_shift,
    plot_substitution_waterfall,
)
from vapor_policy_impact.validation.loso import (
    loso_aggregate_metrics,
    run_loso,
    run_placebo_in_time,
    summarize_loso,
)

---

## Part 2 · Configuration

The only part of this notebook you should need to edit to run against your own data. Leave `USE_SYNTHETIC_DATA = True` to run against the bundled synthetic simulator first — a good way to confirm the notebook runs end-to-end in your environment before pointing it at anything real.


In [ ]:
USE_SYNTHETIC_DATA = True

### 2.1 Data Source

Only used when `USE_SYNTHETIC_DATA = False`. A Delta table, Parquet, or CSV path readable by Spark: a DBFS path, a Unity Catalog Volume path, or `catalog.schema.table` (read via `spark.table()`). `COLUMN_MAPPING` only requires `state`/`week`/`category`/`manufacturer_group`/`volume` — everything else is optional and features degrade gracefully without it (see `vapor_policy_impact/features/engineering.py`).


In [ ]:
DATA_SOURCE_PATH = "/Volumes/catalog/schema/volume/retail_scan_panel"
DATA_SOURCE_FORMAT = "delta"  # "delta" | "parquet" | "csv" | "table"

# our_column_name -> your_column_name.
COLUMN_MAPPING = {
    "state": "STATE_CD",
    "week": "WEEK_END_DATE",        # a real calendar date column, converted automatically below
    "category": "CATEGORY_DESC",
    "manufacturer_group": "MFG_GROUP_DESC",
    "volume": "UNIT_VOLUME",
    # "manufacturer": "MANUFACTURER_NM",
    # "sku": "SKU_ID",
    # "brand": "BRAND_NM",
    # "sales": "DOLLAR_SALES",
    # "price": "AVG_UNIT_PRICE",
    # "promo_depth": "PROMO_PCT_ACV",
    # "distribution_acv": "TOTAL_PCT_ACV",
}
WEEK_IS_DATE_COLUMN = True  # False if your `week` column is already an integer week index

### 2.2 Policy Calendar & Target State

Which states have already implemented the policy, and the state/date you want the 13-week forecast for.


In [ ]:
# state -> policy effective date (or integer week index if WEEK_IS_DATE_COLUMN=False).
# Only include states that have ALREADY implemented the policy -- every other state in
# your data becomes a potential control / synthetic-control donor-pool state.
POLICY_CALENDAR = {
    # "Ohio": "2024-03-01",
    # "Colorado": "2023-11-15",
}

# The state you want the 13-week Policy vs. No-Policy forecast for, and the date its
# policy would take (or took) effect.
TARGET_STATE = "New State"
TARGET_STATE_EFFECTIVE_DATE = "2026-06-01"

# Override the donor pool explicitly, or leave None to default to every state in the
# panel that is neither the target state nor already in POLICY_CALENDAR.
DONOR_POOL_STATES_OVERRIDE = None

### 2.3 State Covariates (Optional)

A small state-level demographics table (population, urbanization, income, border-state flag, retail density — see `pipeline.DEFAULT_COVARIATE_COLS`). Leave `STATE_COVARIATES_PATH = None` to fall back to a minimal covariate set derived from the panel itself, which degrades Layer 3's ability to personalize the transported effect to your target state (see `vapor_policy_impact/data/loaders.py::build_fallback_state_covariates`).


In [ ]:
STATE_COVARIATES_PATH = None
STATE_COVARIATES_FORMAT = "delta"
STATE_COVARIATES_COLUMN_MAPPING = None  # e.g. {"state": "STATE_CD", "population_m": "POP_MILLIONS", ...}

### 2.4 Manufacturer Labels & Model Scope

Which `manufacturer_group` labels in your data correspond to Altria vs. competitors (used by **Part 7**), and how much pre-period history the synthetic-control/baseline forecaster looks back over.


In [ ]:
ALTRIA_LABEL = "Altria"
COMPETITOR_LABEL = "Competitor"

# The framework default is 120 weeks; lower this if your real history is shorter.
PRE_PERIOD_LOOKBACK_WEEKS = 120

### 2.5 Validation & Performance Settings

Bootstrap/Monte-Carlo sizing (lower for faster, noisier iteration; raise for a final run) and whether to run the slow leave-one-state-out validation in **Part 8** — it refits the full stack once per historical treated state, so it's off by default.


In [ ]:
RUN_LOSO_VALIDATION = False  # ~10-15 minutes when True; see Part 8

N_BOOTSTRAP_MAIN = 150
N_BOOTSTRAP_VALIDATION = 60
N_MC = 3000

### 2.6 Databricks Widgets (Optional)


In [ ]:
# A few of the above are also exposed as Databricks widgets for convenience when running
# this notebook as a parameterized job. No-ops outside Databricks.
try:
    dbutils.widgets.dropdown("use_synthetic_data", str(USE_SYNTHETIC_DATA), ["True", "False"])
    dbutils.widgets.text("data_source_path", DATA_SOURCE_PATH)
    dbutils.widgets.text("target_state", TARGET_STATE)
    USE_SYNTHETIC_DATA = dbutils.widgets.get("use_synthetic_data") == "True"
    DATA_SOURCE_PATH = dbutils.widgets.get("data_source_path")
    TARGET_STATE = dbutils.widgets.get("target_state")
except NameError:
    pass

---

## Part 3 · Load Data

Builds the analysis panel, policy calendar, and state covariates — from the synthetic simulator, or from your configuration in **Part 2**.


In [ ]:
if USE_SYNTHETIC_DATA:
    sim = simulate_panel()
    panel = sim.panel
    policy_calendar = sim.policy_calendar
    state_covariates_raw = sim.state_covariates
    target_state = sim.target_state
    target_effective_week = sim.target_state_effective_week
    donor_pool_states = sim.donor_pool_states
    altria_label, competitor_label = "Altria", "Competitor"
    true_no_policy_for_loso = sim.full_ground_truth_no_policy  # ground truth only exists for synthetic data
    print(f"Synthetic panel: {len(panel):,} rows | target state: {target_state} | "
          f"target policy week: {target_effective_week}")

else:
    # --- Real data path -----------------------------------------------------------
    # `spark` is provided automatically in a Databricks notebook. Falls back to a local
    # pandas read so this branch can still be smoke-tested outside Databricks.
    if DATA_SOURCE_FORMAT == "table":
        raw = spark.table(DATA_SOURCE_PATH)
    elif DATA_SOURCE_FORMAT == "csv":
        try:
            raw = spark.read.option("header", True).option("inferSchema", True).csv(DATA_SOURCE_PATH)
        except NameError:
            raw = pd.read_csv(DATA_SOURCE_PATH)
    else:
        try:
            raw = spark.read.format(DATA_SOURCE_FORMAT).load(DATA_SOURCE_PATH)
        except NameError:
            raw = pd.read_parquet(DATA_SOURCE_PATH)

    panel, week_zero_date = load_real_panel(raw, COLUMN_MAPPING, week_is_date=WEEK_IS_DATE_COLUMN)
    policy_calendar = load_policy_calendar(POLICY_CALENDAR, week_zero_date=week_zero_date)

    if STATE_COVARIATES_PATH:
        try:
            cov_source = spark.read.format(STATE_COVARIATES_FORMAT).load(STATE_COVARIATES_PATH)
        except NameError:
            cov_source = pd.read_parquet(STATE_COVARIATES_PATH)
    else:
        cov_source = None
    state_covariates_raw = load_state_covariates(
        cov_source, panel["state"].unique(),
        column_mapping=STATE_COVARIATES_COLUMN_MAPPING, panel_for_fallback=panel,
    )

    target_state = TARGET_STATE
    target_effective_week = (
        date_to_week_index(TARGET_STATE_EFFECTIVE_DATE, week_zero_date)
        if WEEK_IS_DATE_COLUMN else int(TARGET_STATE_EFFECTIVE_DATE)
    )
    donor_pool_states = DONOR_POOL_STATES_OVERRIDE or [
        s for s in panel["state"].unique() if s != target_state and s not in policy_calendar.treated_states
    ]
    altria_label, competitor_label = ALTRIA_LABEL, COMPETITOR_LABEL
    true_no_policy_for_loso = None  # unobservable for real data -- LOSO falls back to actuals-only scoring

    print(f"Real panel: {len(panel):,} rows | states: {panel['state'].nunique()} | "
          f"treated (historical): {list(policy_calendar.treated_states)} | target: {target_state}")

# Defensive, applies in both branches: if the panel happens to include rows for the
# target state at/after its effective week (e.g. you're backtesting against a state
# where the policy already happened), drop them. The baseline forecaster's "last known
# week" must be the last PRE-policy week, exactly like the synthetic simulator gives by
# default -- this exact bug class (forecasting from the wrong starting point) was caught
# during development for the LOSO validation loop; see validation/loso.py::_truncate_panel.
panel = panel[~((panel["state"] == target_state) & (panel["week"] >= target_effective_week))].copy()

categories_to_run = list(CATEGORIES) if USE_SYNTHETIC_DATA else sorted(panel["category"].unique())
covariates = prepare_covariates(state_covariates_raw)

---

## Part 4 · Exploratory Checks

A quick sanity check on what got loaded before running anything expensive: panel shape, state/category coverage, and the target state and event-time window the rest of the notebook will use.


In [ ]:
print("Panel shape:", panel.shape)
display(panel.head())

print("\nStates per category x manufacturer_group:")
display(panel.groupby(["category", "manufacturer_group"])["state"].nunique().rename("n_states"))

print(f"\nTreated (historical) states: {list(policy_calendar.treated_states.keys())}")
print(f"Donor pool ({len(donor_pool_states)} states): {donor_pool_states}")
print(f"Target state: {target_state}  |  target effective week: {target_effective_week}")
print(f"Categories to run: {categories_to_run}")

---

## Part 5 · Causal Modeling & 13-Week Forecast

For each category: a staggered-adoption event study estimates the pooled historical effect curve from your treated states (Layer 1), a heterogeneity model transports it to the target state via its covariates (Layer 3), and a baseline forecaster + Monte Carlo scenario combiner produce the 13-week Policy vs. No-Policy scenarios (methodology sections 4-5, 12).


### 5.1 Run the Pipeline per Category


In [ ]:
category_results = {}
for cat in categories_to_run:
    r = run_category_pipeline(
        panel, policy_calendar, covariates,
        target_state=target_state, target_effective_week=target_effective_week,
        donor_states=donor_pool_states, category=cat,
        pre_period_lookback=PRE_PERIOD_LOOKBACK_WEEKS,
        n_bootstrap=N_BOOTSTRAP_MAIN, n_mc=N_MC,
    )
    category_results[cat] = r
    print(f"{cat:15s} transport_scale={r.transport_scale:+.3f}  "
          f"cum %impact mean={r.scenario.cumulative['pct_impact']['mean']:+.1%}")

### 5.2 Executive Business Table

Methodology section 13.1.


In [ ]:
exec_table = build_executive_table({c: r.scenario for c, r in category_results.items()})
display_executive_table(exec_table)

### 5.3 Event Study


In [ ]:
PLOT_CATEGORY = "Vapor" if "Vapor" in category_results else categories_to_run[0]

fig = plot_event_study(category_results[PLOT_CATEGORY].event_study, title=f"{PLOT_CATEGORY} event study")
show_figure(fig)

### 5.4 13-Week Scenario Forecast


In [ ]:
fig = plot_scenario_fan_chart(category_results[PLOT_CATEGORY].scenario, PLOT_CATEGORY)
show_figure(fig)

---

## Part 6 · Cross-Category Substitution

Reconciles the four category-level estimates into a total-market view and checks whether volume lost from one category shows up as a gain in another (methodology section 6).


### 6.1 Reconciliation & Substitution Waterfall


In [ ]:
recon = reconcile_categories({c: r.scenario for c, r in category_results.items()})
display(recon.category_share_of_gross_movement)
print("Total market (reconciled from category sum):", recon.total_market["pct_impact"])

fig = plot_substitution_waterfall(recon.category_share_of_gross_movement)
show_figure(fig)

### 6.2 Residual-Correlation Diagnostic (Optional)

Needs `price`/`promo_depth`/`distribution_acv` populated to detrend against; skipped gracefully if your real data doesn't have them.


In [ ]:
feat_by_cat = {c: build_aggregated_feature_series(panel, policy_calendar, category=c) for c in categories_to_run}
try:
    corr = category_residual_correlation(feat_by_cat)
    display(corr)
except Exception as e:
    print(f"Skipping residual-correlation diagnostic ({type(e).__name__}: {e}). "
          "This needs price/promo_depth/distribution_acv populated in your panel.")

---

## Part 7 · Altria vs. Competitor Decomposition

Reruns the pipeline split by manufacturer group to isolate Altria's exposure from competitors' and quantify any market-share shift (methodology section 7).


In [ ]:
if {altria_label, competitor_label}.issubset(set(panel["manufacturer_group"].unique())):
    altria_res = run_category_pipeline(
        panel, policy_calendar, covariates, target_state=target_state,
        target_effective_week=target_effective_week, donor_states=donor_pool_states,
        category=PLOT_CATEGORY, manufacturer_group=altria_label,
        pre_period_lookback=PRE_PERIOD_LOOKBACK_WEEKS, n_bootstrap=N_BOOTSTRAP_MAIN, n_mc=N_MC,
    )
    competitor_res = run_category_pipeline(
        panel, policy_calendar, covariates, target_state=target_state,
        target_effective_week=target_effective_week, donor_states=donor_pool_states,
        category=PLOT_CATEGORY, manufacturer_group=competitor_label,
        pre_period_lookback=PRE_PERIOD_LOOKBACK_WEEKS, n_bootstrap=N_BOOTSTRAP_MAIN, n_mc=N_MC,
    )
    decomp = decompose_manufacturer(PLOT_CATEGORY, altria_res.scenario, competitor_res.scenario)

    mfg_table = pd.concat([
        build_executive_table({PLOT_CATEGORY: altria_res.scenario}).assign(View=altria_label),
        build_executive_table({PLOT_CATEGORY: competitor_res.scenario}).assign(View=competitor_label),
    ])
    mfg_table = mfg_table[mfg_table["Metric"] == PLOT_CATEGORY]
    display_executive_table(mfg_table)
    print(f"Reconciled total (sum of manufacturer groups) %impact mean: "
          f"{decomp.total_from_manufacturer_sum['pct_impact']['mean']:+.1%}  "
          f"(category-level estimate was {category_results[PLOT_CATEGORY].scenario.cumulative['pct_impact']['mean']:+.1%})")

    fig = plot_share_shift(decomp.share_shift, PLOT_CATEGORY)
    show_figure(fig)
else:
    print(f"Skipping Altria/competitor decomposition -- manufacturer_group values "
          f"{sorted(panel['manufacturer_group'].unique())} don't include both "
          f"ALTRIA_LABEL={altria_label!r} and COMPETITOR_LABEL={competitor_label!r}. "
          "Set those in Part 2.4 to match your data.")
    altria_res = competitor_res = decomp = None

---

## Part 8 · Model Validation

Two checks on how much to trust the estimates above: leave-one-state-out backtesting against real historical outcomes, and a placebo test that the causal estimator reports ~zero effect when nothing actually happened (methodology section 10).


### 8.1 Leave-One-State-Out Validation

Refits the whole stack once per historical treated state, holding it out, and scores the forecast against that state's *actual* historical post-policy volume. With real data, only forecast-accuracy metrics (WAPE, interval coverage) are computable — the true no-policy counterfactual for a real state is fundamentally unobservable, so the impact-bias/sign-accuracy/rank-correlation metrics are simulation-only and will show as `None`/skipped when `USE_SYNTHETIC_DATA = False`.


In [ ]:
if RUN_LOSO_VALIDATION:
    loso_results = run_loso(
        panel, policy_calendar, covariates, donor_pool_states, category=PLOT_CATEGORY,
        n_bootstrap=N_BOOTSTRAP_VALIDATION, n_mc=1500, true_no_policy=true_no_policy_for_loso,
    )
    loso_summary = summarize_loso(loso_results)
    display(loso_summary)
    print("Aggregate LOSO metrics:", loso_aggregate_metrics(loso_summary))
else:
    print("RUN_LOSO_VALIDATION is False -- skipping (set True in Part 2.5; "
          "expect ~10-15 minutes for a full run across all historical treated states).")
    loso_summary = None

### 8.2 Placebo-in-Time Check

Assigns a fake policy date to genuinely never-treated states and re-runs Layer 1's own estimator — the estimated effect should be statistically indistinguishable from zero. This tests the causal identification itself, independent of any real treated states, and is fast (no baseline forecaster involved).


In [ ]:
PLACEBO_STATES = donor_pool_states[: min(6, len(donor_pool_states))]
if PLACEBO_STATES:
    placebo = run_placebo_in_time(panel, donor_pool_states, PLACEBO_STATES, category=PLOT_CATEGORY, n_bootstrap=200)
    display(placebo)
    print(f"False positive rate at 90% CI: {placebo['false_positive_at_90pct'].mean():.1%} "
          "(expect roughly ~10% under a well-calibrated null; small-N samples are noisy)")
else:
    print("No donor-pool states available for a placebo check.")
    placebo = None

---

## Part 9 · Executive Summary & Productionizing on Databricks


Fill in the executive table and chart references above into your own summary, e.g.:

> Based on the experience of `{n}` states that have already implemented similar vapor
> policies, we project **{target_state}**'s vapor category volume would be **X% lower**
> over the first 13 weeks post-implementation than it would have been otherwise, with
> roughly **Y%** of that loss shifting into cigarettes/MST rather than leaving the
> tracked nicotine category outright.

**Productionizing this notebook on Databricks** (methodology section 8):

- **Feature store**: land the analysis-ready panel as a Delta table, refreshed on your
  normal scan-data cadence; `vapor_policy_impact.features.engineering` stays the single
  source of truth for the transformation logic either way.
- **Model/version lineage**: log each Layer 1/2/3 fit and baseline-forecaster run to
  **MLflow** (parameters: category, target state, bootstrap/MC sizing; artifacts: the
  executive table, event-study and fan-chart figures) so any business output is
  traceable back to the exact model version that produced it.
- **Retraining cadence**: the causal layer (Layers 1-3) only needs periodic retraining
  (e.g. quarterly, or whenever a new state crosses 13+ weeks post-policy) — schedule it
  as a **Databricks Job**. The baseline forecaster should refresh on every new scan-data
  drop for the target state — a separate, more frequent Job.
- **Parameterized re-runs**: the `dbutils.widgets` wired up in Part 2.6 let this
  notebook run as a parameterized Job/Workflow task per target state.
